# 06 多變項分析：調整後風險比與邏輯斯迴歸

Ch05 的分層分析一次只能控制一個干擾因子。這堂課用 **Modified Poisson regression** 同時調整所有因子，
直接算出 **adjusted RR**——與 Ch03/Ch05 一致。同時也用邏輯斯迴歸做對照，
展示高侵襲率下 OR 如何高估效應。

流程：**資料準備 → Crude RR & OR → Adjusted RR (Modified Poisson) → Adjusted OR (Logistic) → 比較 → Forest Plot → 模型診斷**

In [ ]:
# Google Colab setup -- 若在本機執行可跳過此 cell
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || True
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
# === Step 1: 資料準備 + 變項重新編碼 ===

import pathlib
import pandas as pd
import numpy as np
import statsmodels.api as sm               # GLM (Modified Poisson)
import statsmodels.formula.api as smf       # formula API (logistic)
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import warnings

# -- CJK font setup (避免中文標籤顯示為方框) --
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False
plt.style.use("ggplot")
plt.rcParams["figure.dpi"] = 150

# --- 載入資料 ---
df = pd.read_csv("data/synthetic/legionella_outbreak.csv")

# --- 建立二元結果變項 ---
# clinical_severity != 'not_ill' 代表有感染（包含 mild/moderate/severe）
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)

# --- smoking_history 三分類 → 二分類 ---
df["ever_smoker"] = (df["smoking_history"] != "never").astype(int)

# --- 功能狀態轉有序數值 ---
# bedridden(臥床)=0 < assisted(需協助)=1 < independent(獨立行走)=2
fs_map = {"bedridden": 0, "assisted": 1, "independent": 2}
df["functional_score"] = df["functional_status"].map(fs_map)

# --- 快速確認侵襲率 ---
ar = df["infected"].mean()
print(f"全體：{len(df)} 人，感染：{df['infected'].sum()} 人")
print(f"侵襲率 = {ar:.1%}")
print(f"→ 侵襲率 {ar:.0%} 遠高於 10%，OR 會顯著高估效應，應以 RR 為主")

## 為什麼用 Modified Poisson 算 RR？

- Ch03：世代研究 → 效應測量用 **RR**
- Ch05：分層分析 → **MH adjusted RR**
- 本章：多變項分析 → **adjusted RR**（Modified Poisson）

**Modified Poisson（Zou 2004）**：用 Poisson GLM + robust sandwich SE，coefficient = log(RR)。
就像跟朋友借了一頂帽子（Poisson 是給計數資料用的），尺寸不對但貼個修正貼紙（robust SE）就完美合頭了。

同時也會跑 **邏輯斯迴歸**（→ OR），讓你看到高侵襲率下 OR 高估多少。

In [ ]:
# === Step 2: 單變項分析 — Crude RR 與 Crude OR 對照 ===
# 同時跑 Modified Poisson（RR）和 logistic（OR），看同一變項的差異

from epi_learning.metrics import risk_ratio  # Ch03 的 2×2 手算 RR

factors = [
    "shower_use", "hydrotherapy_use", "ever_smoker",
    "comorbidity_chf", "comorbidity_dm", "comorbidity_cancer",
    "comorbidity_copd", "immunosuppressed",
    "age", "functional_score",
]

crude_results = []

for var in factors:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")

        # (A) Modified Poisson → crude RR
        try:
            mod_p = smf.glm(
                f"infected ~ {var}", data=df,
                family=sm.families.Poisson(),
            ).fit(cov_type="HC0", disp=0)
            rr = np.exp(mod_p.params[var])
            rr_ci = np.exp(mod_p.conf_int().loc[var])
        except Exception:
            continue

        # (B) Logistic → crude OR
        try:
            mod_l = smf.logit(f"infected ~ {var}", data=df).fit(disp=0, method="lbfgs")
            if not mod_l.mle_retvals["converged"]:
                print(f"⚠ {var}: logistic 未收斂，跳過")
                continue
            or_val = np.exp(mod_l.params[var])
            or_ci = np.exp(mod_l.conf_int().loc[var])
        except Exception:
            continue

    # (C) 2×2 手算 RR 交叉驗證（僅二元變項）
    hand_rr = ""
    if df[var].dropna().isin([0, 1]).all():
        a = ((df[var] == 1) & (df["infected"] == 1)).sum()
        b = ((df[var] == 1) & (df["infected"] == 0)).sum()
        c = ((df[var] == 0) & (df["infected"] == 1)).sum()
        d = ((df[var] == 0) & (df["infected"] == 0)).sum()
        hand_rr = f"{risk_ratio(a, a+b, c, c+d):.3f}"

    crude_results.append({
        "variable": var,
        "crude_RR": round(rr, 3),
        "RR 95% CI": f"{rr_ci[0]:.3f}\u2013{rr_ci[1]:.3f}",
        "crude_OR": round(or_val, 3),
        "OR 95% CI": f"{or_ci[0]:.3f}\u2013{or_ci[1]:.3f}",
        "hand_RR": hand_rr,
    })

crude_df = pd.DataFrame(crude_results)
print("=== Crude RR vs Crude OR（單變項）===")
print(crude_df.to_string(index=False))
print()
print("💡 crude_OR 普遍大於 crude_RR — 高侵襲率下 OR 高估的效果")
print("   hand_RR 欄 = Ch03 的 2×2 手算，應與 crude_RR 幾乎一致")

### 讀懂公式語法

statsmodels 的公式借用了 R 語言的 **formula 語法**，用一行字描述「用哪些變項來預測結果」：

| 符號 | 意思 | 範例 |
|------|------|------|
| `~` | 「被⋯預測」 | `infected ~ age` → 用 age 預測 infected |
| `+` | 「再加上」 | `~ age + sex` → 同時放 age 和 sex 進模型 |
| `C()` | 「當成類別變項」 | `C(floor)` → 把 floor 拆成虛擬變項（dummy coding），每個樓層一個 0/1 指標 |

白話文：`infected ~ shower_use + age + C(floor)` 就是說「用淋浴使用、年齡、樓層來預測感染」。模型會自動加上截距項（Intercept），不用額外寫。

### 模型放哪些變項？——從 Ch03 和 Ch05 的結果出發

多變項模型不是把所有欄位都丟進去，而是要有理由。回顧前面章節的發現，我們把變項分成四組：

| 組別 | 變項 | 角色 | 納入理由 |
|------|------|------|----------|
| **暴露因子** | `shower_use`, `hydrotherapy_use` | 研究焦點 | Ch03 篩選出的顯著危險因子——我們最想回答的問題：「淋浴和水療是不是感染源？」 |
| **宿主因子** | `age`, `immunosuppressed`, `functional_score` | 潛在干擾因子 | `age` = 流行病學常規必調整因子；`immunosuppressed` = Ch03 顯示 crude RR 最高的因子之一；`functional_score` = Ch05 已確認的干擾因子 |
| **共病** | `comorbidity_chf/dm/cancer/copd` | 潛在干擾因子 | Ch03 篩選出的候選因子，放入完整模型看控制後暴露因子的 RR 是否改變 |
| **場所** | `C(floor)` | 潛在干擾因子 | 不同樓層的水管系統或暴露機會可能不同，需要控制樓層差異 |

> **為什麼不放 `ever_smoker`？** Ch03 的單變項篩選中，吸菸的 crude RR 接近 1 且未達統計顯著，加上它與多項共病高度相關（共線性），納入模型反而增加估計的不穩定性，因此不放進多變項模型。

In [ ]:
# === Step 3: Modified Poisson — 多變項 Adjusted RR ===
# 本章主軸分析：Poisson GLM + robust SE → coefficient = log(RR)

# --- 公式說明 ---
# infected ~ ：用右邊的變項預測「是否感染」
# shower_use + hydrotherapy_use ：暴露因子（研究焦點）
# age + immunosuppressed + functional_score ：宿主因子（潛在干擾）
# comorbidity_chf/dm/cancer/copd ：共病（潛在干擾）
# C(floor) ：樓層當類別變項（控制場所差異）
formula = (
    "infected ~ shower_use + hydrotherapy_use + age + "
    "comorbidity_chf + comorbidity_dm + comorbidity_cancer + "
    "comorbidity_copd + immunosuppressed + functional_score + "
    "C(floor)"     # C(floor) = 把 floor 當成類別變項（dummy coding）
)

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    model_poisson = smf.glm(
        formula, data=df,
        family=sm.families.Poisson(),   # Poisson 的殼
    ).fit(cov_type="HC0")               # robust (sandwich) SE

# --- 整理成 Table 2 格式 ---
adj_rr_results = []
for var in model_poisson.params.index:
    if var == "Intercept":
        continue
    coef = model_poisson.params[var]
    ci = model_poisson.conf_int().loc[var]
    adj_rr_results.append({
        "variable": var,
        "adjusted_RR": round(np.exp(coef), 3),
        "95% CI": f"{np.exp(ci[0]):.3f}\u2013{np.exp(ci[1]):.3f}",
        "p-value": round(model_poisson.pvalues[var], 4),
    })

adj_rr_df = pd.DataFrame(adj_rr_results)
print("=== Adjusted RR（Modified Poisson, Table 2）===")
print(adj_rr_df.to_string(index=False))

In [ ]:
# === Step 4: Logistic Regression — 多變項 Adjusted OR（對照）===
# 同一公式，改用 logistic regression，看 OR 比 RR 高估多少

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    model_logit = smf.logit(formula, data=df).fit(disp=0, method="lbfgs")

adj_or_results = []
for var in model_logit.params.index:
    if var == "Intercept":
        continue
    coef = model_logit.params[var]
    ci = model_logit.conf_int().loc[var]
    adj_or_results.append({
        "variable": var,
        "adjusted_OR": round(np.exp(coef), 3),
        "95% CI": f"{np.exp(ci[0]):.3f}\u2013{np.exp(ci[1]):.3f}",
        "p-value": round(model_logit.pvalues[var], 4),
    })

adj_or_df = pd.DataFrame(adj_or_results)
print("=== Adjusted OR（Logistic Regression, Table 2）===")
print(adj_or_df.to_string(index=False))

In [ ]:
# === Step 5: Crude RR vs Adjusted RR vs Adjusted OR 並排比較 ===

key_vars = ["shower_use", "hydrotherapy_use", "age",
            "comorbidity_chf", "immunosuppressed", "functional_score"]

comparison = []
for var in key_vars:
    c_row = crude_df[crude_df["variable"] == var]
    a_rr_row = adj_rr_df[adj_rr_df["variable"] == var]
    a_or_row = adj_or_df[adj_or_df["variable"] == var]
    if len(c_row) == 0 or len(a_rr_row) == 0 or len(a_or_row) == 0:
        continue
    c_rr = c_row.iloc[0]["crude_RR"]
    a_rr = a_rr_row.iloc[0]["adjusted_RR"]
    a_or = a_or_row.iloc[0]["adjusted_OR"]
    rr_chg = ((a_rr - c_rr) / c_rr * 100) if c_rr != 0 else 0
    or_vs = ((a_or - a_rr) / a_rr * 100) if a_rr != 0 else 0
    comparison.append({
        "variable": var,
        "crude_RR": c_rr,
        "adj_RR": a_rr,
        "adj_OR": a_or,
        "crude→adj RR": f"{rr_chg:+.1f}%",
        "adj RR→OR": f"{or_vs:+.1f}%",
    })

comp_df = pd.DataFrame(comparison)
print("=== Crude RR → Adjusted RR → Adjusted OR 比較 ===")
print(comp_df.to_string(index=False))
print()
print("📊 crude→adj RR：控制干擾因子後 RR 的變化")
print("   adj RR→OR：同一模型下 OR 比 RR 高估多少")

In [ ]:
# === Step 6: Forest Plot — Adjusted RR ===

plot_vars = [r for r in adj_rr_df["variable"] if not r.startswith("C(floor)")]
plot_data = adj_rr_df[adj_rr_df["variable"].isin(plot_vars)].copy()
plot_data["ci_lo"] = plot_data["95% CI"].str.split("\u2013").str[0].astype(float)
plot_data["ci_hi"] = plot_data["95% CI"].str.split("\u2013").str[1].astype(float)

fig, ax = plt.subplots(figsize=(8, 5))
y_pos = range(len(plot_data))
ax.errorbar(
    plot_data["adjusted_RR"], y_pos,
    xerr=[plot_data["adjusted_RR"] - plot_data["ci_lo"],
          plot_data["ci_hi"] - plot_data["adjusted_RR"]],
    fmt="o", color="#D97757", ecolor="#6A9BCC",
    elinewidth=2, capsize=4, markersize=7,
)
ax.axvline(x=1, color="#6B6B6B", linestyle="--", linewidth=1, label="RR = 1")
ax.set_yticks(list(y_pos))
ax.set_yticklabels(plot_data["variable"])
ax.set_xlabel("Adjusted RR（95% CI）")
ax.set_title("Forest Plot — Adjusted Risk Ratio（Modified Poisson）")
ax.legend(loc="lower right", fontsize=9)
ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# === Step 7: 模型診斷 — AIC 比較 ===
# 用 AIC 比較「完整模型」和「精簡模型」，判斷是否放了太多變項。

# --- 精簡模型 ---
# 移除 Ch03 篩選中 crude RR 不顯著或效果量小的共病，以及樓層。
# 保留：核心暴露因子（shower_use, hydrotherapy_use）
#       + 理論上最重要的調整因子（age, immunosuppressed, functional_score）
formula_reduced = (
    "infected ~ shower_use + hydrotherapy_use + age + "
    "immunosuppressed + functional_score"
)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    model_reduced = smf.glm(
        formula_reduced, data=df,
        family=sm.families.Poisson(),
    ).fit(cov_type="HC0")

print("=== 模型比較（Modified Poisson）===")
print(f"  完整模型 AIC = {model_poisson.aic:.1f}")
print(f"  精簡模型 AIC = {model_reduced.aic:.1f}")
print()
if model_reduced.aic < model_poisson.aic:
    print("📉 精簡模型 AIC 較小 → 精簡模型取得更好平衡")
else:
    print("📈 完整模型 AIC 較小 → 多放的變項確實有貢獻")

### 變項怎麼選？三種策略比較

上面只比了「完整 vs 精簡」兩個模型。但實務上，到底要放哪些變項進模型？有三種常見策略：

| 策略 | 做法 | 優點 | 缺點 |
|------|------|------|------|
| **Forward（往前加）** | 從空模型開始，每次加入一個讓 AIC 下降最多的變項 | 簡單直覺 | 容易漏掉聯合效應；加入順序影響結果 |
| **Backward（往後刪）** | 從完整模型開始，每次移除一個對 AIC 影響最小的變項 | 能看到所有變項的聯合效應 | 需要夠多樣本才能放所有變項 |
| **Change-in-estimate（效應改變法）** | 逐一移除候選干擾因子，看暴露因子的 RR 是否改變 ≥ 10% | **流行病學金標準**——以「是否干擾暴露效應」為判斷依據 | 需要先定義「暴露因子」 |

**流行病學推薦用 change-in-estimate**，而不是 stepwise（自動選變項）。原因很簡單：我們做多變項分析的目的是**正確估計暴露因子的效應**，不是做預測。一個變項即使 p-value 不顯著，只要它是干擾因子（移除後讓 RR 改變 ≥ 10%），就應該留在模型裡。

下面用 Python 實作 change-in-estimate 法：

In [ ]:
# === Step 7b: Change-in-Estimate 變項選擇 ===
# 流行病學標準做法：逐一移除候選干擾因子，
# 看暴露因子（shower_use, hydrotherapy_use）的 adjusted RR 改變多少。
# 改變 ≥ 10% → 該變項是干擾因子，必須留在模型裡。

# --- 完整模型的暴露因子 RR（基準值）---
full_rr = {
    var: np.exp(model_poisson.params[var])
    for var in ["shower_use", "hydrotherapy_use"]
}
print("完整模型的暴露因子 RR（基準）：")
for var, rr in full_rr.items():
    print(f"  {var}: {rr:.3f}")
print()

# --- 候選干擾因子：逐一移除測試 ---
confounders = [
    "age", "comorbidity_chf", "comorbidity_dm", "comorbidity_cancer",
    "comorbidity_copd", "immunosuppressed", "functional_score", "C(floor)",
]

cie_results = []
for drop_var in confounders:
    # 建立移除一個變項的公式
    keep = [c for c in confounders if c != drop_var]
    formula_test = "infected ~ shower_use + hydrotherapy_use + " + " + ".join(keep)

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        m = smf.glm(formula_test, data=df, family=sm.families.Poisson()).fit(
            cov_type="HC0", disp=0
        )

    for exposure in ["shower_use", "hydrotherapy_use"]:
        rr_without = np.exp(m.params[exposure])
        pct_change = (rr_without - full_rr[exposure]) / full_rr[exposure] * 100
        cie_results.append({
            "移除的變項": drop_var,
            "暴露因子": exposure,
            "移除後 RR": round(rr_without, 3),
            "RR 改變%": f"{pct_change:+.1f}%",
            "是否干擾": "✓ 干擾" if abs(pct_change) >= 10 else "",
        })

cie_df = pd.DataFrame(cie_results)
print("=== Change-in-Estimate 分析 ===")
print("（移除某變項後，暴露因子的 RR 改變 ≥ 10% → 該變項是干擾因子）\n")
print(cie_df.to_string(index=False))
print()
print("📋 看「RR 改變%」欄位：改變 ≥ 10% 的變項是干擾因子，")
print("   即使它自己的 p-value 不顯著，也必須留在模型裡。")

## 小結

| 步驟 | 方法 | 輸出 |
|------|------|------|
| Crude RR | Modified Poisson 單變項 | crude RR |
| Adjusted RR | Modified Poisson 多變項 | **adjusted RR**（主軸）|
| Adjusted OR | Logistic Regression 多變項 | adjusted OR（對照）|
| 並排比較 | 三欄表格 | crude→adj 干擾效應 + OR 高估幅度 |
| 森林圖 | matplotlib errorbar | 視覺化 adjusted RR |
| 模型診斷 | AIC 比較 | 完整 vs 精簡 |

**結論**：Modified Poisson 是世代研究中多變項分析的首選方法（直接算 RR）。
邏輯斯迴歸在高侵襲率下會高估效應（OR > RR），但在病例對照研究或罕見疾病中仍是標準方法。

下一章（Ch07）：主管想知道「下週還會有多少新個案？」→ 時間序列預測